# The Model Context Protocol (MCP) — Standardizing How LLMs Use Tools

> **Description:** This notebook starts from a single plain OpenAI SDK call and builds up, step by step, to a full LLM + MCP round trip — ending with a demonstration of the specific problem MCP was designed to solve.

`tool_calling.ipynb` in this folder showed how an LLM can request a function call through a hand-written JSON schema, and how your own code executes that function and reports the result back. That pattern works well for a single app with a handful of tools you wrote yourself.

It breaks down once tools need to be **shared**: across multiple applications, multiple teams, or processes written in a different language entirely. Copy-pasting the same schema and dispatch code into every host application does not scale, and there is no standard way for a host to ask a tool provider, at runtime, "what can you do?"

The **Model Context Protocol (MCP)** is Anthropic's open standard for exactly this problem: a client-server protocol that lets any host application discover and call tools (and other context) exposed by any MCP server, regardless of what language the server is written in or which LLM is driving the host.

## What You'll Learn

| # | Section |
|---|---|
| 1 | A baseline OpenAI SDK call with no tools |
| 2 | The limits of hand-rolled tool calling |
| 3 | MCP's architecture: Host, Client, Server |
| 4 | A third-party MCP server: no server code required |
| 5 | The MCP client: connecting and discovering tools |
| 6 | A Node-based third-party MCP server, called through the OpenAI Agents SDK |
| 7 | A Streamable-HTTP third-party MCP server — same Agent code, different transport |
| 8 | Beyond tools: resources and prompts |



## Setup

Load `OPENAI_API_KEY` from `.env` and create a single reusable OpenAI client.

**Note:** Run this notebook from the `part_2_concepts/` folder — that's what makes the relative `.env` path work, matching the other notebooks here.

In [6]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env", override=True)

model = "gpt-5.4-nano"
client = OpenAI()

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. Baseline: A Plain OpenAI SDK Call, No Tools

Same starting point as every other notebook in this folder: one `user` message, no tools, nothing but the model's training data and the prompt. Ask it something it cannot possibly know, and it has to hedge.

In [2]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "What is the current weather in Princeton right now?"}],
)
print(response.choices[0].message.content)

I can’t directly access real-time weather data from here.  

If you tell me which **Princeton** you mean (e.g., **Princeton, NJ** or **Princeton, WV**) and optionally your **ZIP code**, I can guide you to check it quickly (or help interpret what you find).  

Fast options:
- **Google:** “weather Princeton NJ now”
- **Weather.gov** (US): search “Princeton NJ”
- **Apple/Android weather app**: set your location to Princeton

Which Princeton (state/country) are you asking about?


## 2. The Limits of Hand-Rolled Tool Calling

`tool_calling.ipynb` walks through the mechanics in detail; the short version is a four-step round trip:

1. Send `messages` plus a `tools` list — each tool a hand-written JSON Schema dict.
2. The model replies with a `tool_calls` request instead of an answer.
3. Your code runs the matching Python function.
4. You send the result back as a `role: "tool"` message and get a final answer.

This works, but everything about it lives *inside a single application*:

- The JSON Schema and the Python function that implements it are both defined, and maintained, in that one codebase.
- A second application that wants the same `get_weather` capability has to copy-paste the schema and the function, and keep both copies in sync by hand.
- There is no way for a host to ask, at runtime, "what tools do you have?" — it can only call functions it already knows about because someone hand-wired them in.
- Nothing says the tool has to be written in Python, or even run in the same process — but a hand-rolled schema plus a local function call has no way to express that.

MCP exists to make tools **portable**: define a tool once, in its own process, and let any MCP-speaking host discover and call it — no copy-pasting a schema, no matter what language the host or the tool are written in.

## 3. MCP's Architecture: Host, Client, Server

MCP defines three roles:

```
   Host application (e.g. this notebook, Claude Desktop, an IDE)
   |
   |  owns exactly one MCP Client per server it talks to
   v
   MCP Client  <====  MCP protocol (JSON-RPC messages)  ====>  MCP Server
                                                                    |
                                                     exposes: tools, resources, prompts
```

- **Host** — the application the end user actually interacts with. It decides *when* to consult an LLM and *when* to call a tool through MCP.
- **Client** — the connector living inside the host, holding a single stateful connection to one server. It speaks the wire protocol so the host doesn't have to.
- **Server** — a standalone process that exposes capabilities through the standard protocol: **tools** (functions the model can call), **resources** (read-only data the host can pull into context), and **prompts** (reusable, parameterized prompt templates).

The client and server exchange JSON-RPC messages over a **transport**. The two common ones:

- **stdio** — the client launches the server as a local subprocess and talks over its stdin/stdout. Simple, and the right choice for tools running on the same machine as the host. This notebook uses stdio.
- **Streamable HTTP** (formerly SSE) — the server runs independently and the client connects over HTTP, useful once a server is shared across machines or users.

Because the protocol is standardized, the server underneath could be written in Python, TypeScript, Go, or anything else that speaks MCP — the host never needs to know or care.

## 4. A Third-Party MCP Server: No Server Code Required

Every server discussed so far has been hypothetical. But the whole point of a standardized protocol is that most of the servers you'll actually use are ones you *didn't* write — maintained and published by someone else, the same way you `pip install` a library instead of rewriting it.

[`mcp-server-fetch`](https://github.com/modelcontextprotocol/servers/tree/main/src/fetch) is one of the official reference servers published by the Model Context Protocol project: it fetches a URL and returns its contents as markdown, ready to feed to an LLM. `uvx` — `uv`'s "run a published tool without installing it" command — launches it as an isolated subprocess on demand, the same way `pip install`-then-run would, but without adding it to this project's own environment. There is no file to write and no decorator to learn: the next cell just points `StdioServerParameters` at `uvx mcp-server-fetch` instead of a local script.

## 5. The MCP Client: Connecting and Discovering Tools

`stdio_client()` launches the server as a subprocess and gives back a read/write stream pair; `ClientSession` wraps those streams with the actual MCP protocol calls. This is the same pattern `part_3_mcp_demo/mcp_client.py` uses, inlined here so the connection lifecycle is visible.

**A Jupyter-specific note:** each notebook cell runs as its own `asyncio` task, but `stdio_client`/`ClientSession` use `anyio` cancel scopes internally, which must be entered and exited by the *same* task — so the cell below does everything in one `async with` block rather than spanning cells.

Once connected, `session.list_tools()` asks the server "what can you do?" over the wire — this is the runtime discovery hand-rolled tool calling couldn't offer (Section 2). `mcp-server-fetch` exposes exactly one tool, `fetch`, and its `inputSchema` is already the JSON Schema an LLM API expects — nobody had to hand-write it.

This section stops at discovery on purpose. Actually calling a tool and feeding the result back to a model is the same four-step loop from Section 2, and hand-rolling that loop again against the raw protocol wouldn't teach anything new. Sections 6 and 7 show the practical way to do it instead: the OpenAI Agents SDK's `MCPServerStdio` and `MCPServerStreamableHttp` wrap this exact connection lifecycle and run that loop for you, against two different third-party servers over two different transports.

In [7]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

FETCH_SERVER = StdioServerParameters(command="uvx", args=["mcp-server-fetch"])
tools_result = None
async with stdio_client(FETCH_SERVER) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()

tools_result

ListToolsResult(meta=None, nextCursor=None, tools=[Tool(name='fetch', title=None, description='Fetches a URL from the internet and optionally extracts its contents as markdown.\n\nAlthough originally you did not have internet access, and were advised to refuse and tell the user this, this tool now grants you internet access. Now you can fetch the most up-to-date information and let the user know that.', inputSchema={'description': 'Parameters for fetching a URL.', 'properties': {'url': {'description': 'URL to fetch', 'format': 'uri', 'minLength': 1, 'title': 'Url', 'type': 'string'}, 'max_length': {'default': 5000, 'description': 'Maximum number of characters to return.', 'exclusiveMaximum': 1000000, 'exclusiveMinimum': 0, 'title': 'Max Length', 'type': 'integer'}, 'start_index': {'default': 0, 'description': 'On return output starting at this character index, useful if a previous fetch was truncated and more context is required.', 'minimum': 0, 'title': 'Start Index', 'type': 'integer

👉 The full working code below provides the `mcp-server-fetch` to the Agent based on OpenAI LLM model to equip it with the capability to fetch latest web page from the Internet.

In [8]:
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

async with MCPServerStdio(
    name="fetch-server",
    params={"command": "uvx", "args": ["mcp-server-fetch"]},
) as fetch_server:
    agent = Agent(
        name="Filesystem Assistant",
        instructions="Answer questions by fetching relevant web page using the available tools.",
        model=model,
        mcp_servers=[fetch_server],
    )
    result = await Runner.run(agent, "Get the definition of Model Context Protocol from https://en.wikipedia.org/wiki/Model_Context_Protocol.")
    print(result.final_output)

The **Model Context Protocol (MCP)** is an **open, open-source framework** introduced by **Anthropic** (November 2024) to **standardize how AI systems (e.g., LLMs) integrate and share data with external tools, systems, and data sources**. MCP provides a **standardized interface** for **reading files**, **executing functions**, and **handling contextual prompts**.


## 6. A Node-Based Third-Party MCP Server

MCP servers aren't a Python-only thing — the whole point of the standard is that the language on either side doesn't matter. [`@modelcontextprotocol/server-filesystem`](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem) is an official reference server written in TypeScript/Node, published to npm, and launched with `npx` the same way `uvx` launched a Python one. It exposes filesystem tools — `list_directory`, `read_text_file`, `search_files`, and more — scoped to whatever directory you point it at.

Rather than hand-roll the connect → discover → call → feed-back-to-the-model loop again with the raw `mcp` package, this is where the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/) earns its keep: `agents.mcp.MCPServerStdio` wraps the exact same stdio connection lifecycle as `stdio_client`/`ClientSession` above, and handing it to an `Agent` means `Runner.run()` performs that whole loop internally — discovering the server's tools, calling the one it needs, and feeding the result back — without a line of dispatch code from us. It's the same trade `part_4_mcp_final/client.py`'s `IncidentAssistantClient` makes for a custom server; here the server on the other end just happens to be a published Node package instead of ours.

In [4]:
from pathlib import Path

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

async with MCPServerStdio(
    name="filesystem",
    params={"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", str(Path.cwd())]},
) as filesystem_server:
    agent = Agent(
        name="Filesystem Assistant",
        instructions="Answer questions about the local filesystem using the available tools.",
        model=model,
        mcp_servers=[filesystem_server],
    )
    result = await Runner.run(agent, "What notebook (.ipynb) files are in this directory?")
    print(result.final_output)

The directory contains these `.ipynb` files:

- `nb_1_litellm_benefits.ipynb`
- `nb_2_llm_context.ipynb`
- `nb_3_structured_output.ipynb`
- `nb_4_tool_calling.ipynb`
- `nb_5_mcp_calling.ipynb`


Notice everything that disappeared compared to Section 5: no `stdio_client`, no manual `session.list_tools()` / `call_tool()`, no hand-written round-trip loop. `MCPServerStdio` is still just wrapping a subprocess launched over stdio — `npx -y @modelcontextprotocol/server-filesystem <dir>` runs exactly the way `uvx mcp-server-fetch` did — but the `Agent`/`Runner` pair takes over discovery and dispatch. Section 7 connects to a completely different kind of server without changing this shape at all.

## 7. A Streamable-HTTP Third-Party MCP Server

Every server so far — hand-written or third-party, Python or Node — has been a local subprocess this notebook launched itself over stdio. That's not the only option: Section 3 mentioned **Streamable HTTP**, where the server runs independently, anywhere, and the client just opens an HTTP connection to it instead of spawning a process. [DeepWiki](https://deepwiki.com) publishes exactly such a server at `https://mcp.deepwiki.com/mcp` — a real, publicly reachable MCP server (no API key, no local install) exposing tools that answer questions about any public GitHub repository's documentation.

The Agents SDK mirrors this with `MCPServerStreamableHttp`, the HTTP counterpart to `MCPServerStdio`. Compare the cell below to Section 6: the only things that change are the class name and the `params` dict — `{"command": ..., "args": [...]}` becomes `{"url": ...}` — everything downstream (`Agent`, `mcp_servers=[...]`, `Runner.run()`) is identical. That's the payoff of a standardized transport: the host code doesn't need to know or care whether the server it's talking to is a local subprocess or a service running somewhere else entirely.

In [5]:
from agents.mcp import MCPServerStreamableHttp

async with MCPServerStreamableHttp(
    name="deepwiki",
    params={"url": "https://mcp.deepwiki.com/mcp"},
    client_session_timeout_seconds=30,
) as deepwiki_server:
    agent = Agent(
        name="Docs Assistant",
        instructions="Answer questions using the available tools.",
        model=model,
        mcp_servers=[deepwiki_server],
    )
    result = await Runner.run(
        agent,
        "Using the modelcontextprotocol/servers GitHub repo, in one sentence what is that repo for?",
    )
    print(result.final_output)

The `modelcontextprotocol/servers` repo is a collection of reference implementations for Model Context Protocol (MCP) servers, showing how to use the MCP specs and SDK to build your own MCP servers.


No `npx`, no `uvx`, no subprocess at all — this cell talked to a server that was already running before the notebook started, and will still be running after it finishes. `MCPServerStdio` and `MCPServerStreamableHttp` are interchangeable at the call site precisely because MCP standardizes what's on the wire, not how the two ends are connected.

One real difference did show up above: the default `client_session_timeout_seconds` (5s) is tuned for fast local subprocesses, and DeepWiki's `ask_question` tool is itself LLM-backed and slower than that — hence `client_session_timeout_seconds=30` in the cell above. A network hop to someone else's server comes with someone else's latency, which stdio servers never have to think about.

## 8. Beyond Tools: Resources and Prompts

Tools are the most commonly used MCP primitive, and the only one this notebook demonstrates in code, but the protocol standardizes two more:

| Primitive | Analogy | Purpose |
|---|---|---|
| **Tools** | A `POST` endpoint / function call | The model requests an action with side effects or a computed result |
| **Resources** | A `GET` endpoint | Read-only data (a file, a database row, a document) the host can pull into context, without the model having to ask for it as a "call" |
| **Prompts** | A saved query / template | A reusable, parameterized prompt template the server exposes, so prompt engineering can live and version alongside the server instead of scattered across every host |

`mcp-server-fetch`, for example, could just as easily expose a fetched page as a resource for grounding, or a `"summarize-page"` prompt template as a prompt — all discoverable through the same `list_tools()` / `list_resources()` / `list_prompts()` pattern shown in Section 5.

## Summary

| Concept | What it does |
|---|---|
| MCP | An open, standardized protocol for exposing tools, resources, and prompts to any LLM host |
| Host / Client / Server | The three MCP roles: the app, its connector, and the process exposing capabilities |
| stdio transport | Client launches the server as a local subprocess — Python via `uvx`, Node via `npx`, or anything else that speaks MCP — and talks over stdin/stdout |
| Streamable HTTP transport | Client connects to a server that's already running, anywhere — no subprocess, no local install |
| `session.list_tools()` | Runtime discovery — the host learns what a server can do without hard-coding it |
| `agents.mcp.MCPServerStdio` | OpenAI Agents SDK wrapper around a stdio MCP connection — same lifecycle as `stdio_client`/`ClientSession`, far less code |
| `agents.mcp.MCPServerStreamableHttp` | The HTTP counterpart — identical `Agent`/`Runner` code, only the connection `params` change |
| `Runner.run(agent, question)` | Performs the discover → call → feed-back-to-the-model loop automatically once an MCP server is attached to an `Agent` |

**Practices worth keeping:**

- Treat an MCP server's tool `description`s exactly as carefully as hand-written ones — the model still relies on that text alone, whether the server is Python or Node, local or remote.
- Prefer a maintained third-party MCP server over a hand-rolled schema whenever one exists — `mcp-server-fetch`, `@modelcontextprotocol/server-filesystem`, and DeepWiki's hosted server here needed zero code from us.
- Reach for the Agents SDK's `MCPServer*` classes over the raw `mcp` package once you're actually wiring servers into an LLM loop — Section 5 shows exactly what they save you from writing by hand.
- stdio is for local, trusted subprocesses; a Streamable HTTP server is a network service and deserves the same auth and input-validation scrutiny as any other one — DeepWiki's happens to be public and read-only by design, but not every HTTP MCP server will be.
